# **Day 63: Regular Expressions (Part 2) - Checking Real Formats**

Today we use regex to **validate** (check) real formats from India, West Bengal and Remote Sensing.

## **Named Groups**
`(?P<name>...)` gives a **name** to a group, so the result is easier to read.

In [1]:
import re

vehicle = re.compile(r"^(?P<state>WB)[ -]?(?P<rto>\d{2})[ -]?(?P<series>[A-Z]{1,2})[ -]?(?P<number>\d{4})$", re.I)

for v in ["WB 02 AB 1234", "wb74c0042", "WB-06-AK-9876", "OD 02 AB 1234", "WB 2 A 12"]:
    m = vehicle.match(v)
    print(f"{v:<15}", "valid  " + str(m.groupdict()) if m else "invalid")

WB 02 AB 1234   valid  {'state': 'WB', 'rto': '02', 'series': 'AB', 'number': '1234'}
wb74c0042       valid  {'state': 'wb', 'rto': '74', 'series': 'c', 'number': '0042'}
WB-06-AK-9876   valid  {'state': 'WB', 'rto': '06', 'series': 'AK', 'number': '9876'}
OD 02 AB 1234   invalid
WB 2 A 12       invalid


On Day 38 we did the same check with `split()` and many `if` statements. Regex does it in one pattern.

## **Indian Mobile Numbers**
10 digits, starting with 6, 7, 8 or 9, with an optional `+91` or `0` in front.

In [2]:
mobile = re.compile(r"^(?:\+91[\s-]?|0)?([6-9]\d{9})$")

for num in ["9830012345", "+91 9433012345", "09051234567", "5830012345", "+91-7001234567"]:
    m = mobile.match(num)
    print(f"{num:<16}", "OK -> " + m.group(1) if m else "invalid")

9830012345       OK -> 9830012345
+91 9433012345   OK -> 9433012345
09051234567      OK -> 9051234567
5830012345       invalid
+91-7001234567   OK -> 7001234567


- `(?:...)` is a group that does **not** capture. We use it only to make the prefix optional with `?`.

## **West Bengal PIN Codes and Email**

In [3]:
wb_pin = re.compile(r"^7[0-4]\d{4}$")
print([p for p in ["700001", "721101", "734001", "751001", "7000011"] if wb_pin.match(p)])

email = re.compile(r"^[\w.+-]+@[\w-]+(\.[\w-]+)+$")
for e in ["tublu.das@gmail.com", "info@kolkata.gov.in", "bad@", "no_at_sign.com"]:
    print(f"{e:<22}{bool(email.match(e))}")

['700001', '721101', '734001']
tublu.das@gmail.com   True
info@kolkata.gov.in   True
bad@                  False
no_at_sign.com        False


## **Example from GIS: Reading Coordinates Written in Many Styles**
People write coordinates in many different ways:
- `22.5726, 88.3639`
- `22.5726N 88.3639E`
- `Lat: 22.5726 Lon: 88.3639`
- `22°34'21"N 88°21'50"E`

One regex for decimal numbers, and another for DMS, can read all of them.

In [4]:
decimal_pair = re.compile(r"(-?\d{1,3}\.\d+)\s*[NS]?[,\s]+(?:Lon:\s*)?(-?\d{1,3}\.\d+)\s*[EW]?")
dms = re.compile(r"(\d{1,3})°(\d{1,2})'(\d{1,2}(?:\.\d+)?)\"([NSEW])")

def read_coordinate(text):
    found = dms.findall(text)
    if len(found) == 2:
        values = []
        for d, m, s, hemi in found:
            dd = int(d) + int(m) / 60 + float(s) / 3600
            values.append(-dd if hemi in "SW" else dd)
        return round(values[0], 5), round(values[1], 5)
    m = decimal_pair.search(text)
    if m:
        return float(m.group(1)), float(m.group(2))
    return None

samples = ["22.5726, 88.3639", "22.5726N 88.3639E", "Lat: 22.5726 Lon: 88.3639",
           "22°34'21\"N 88°21'50\"E", "somewhere near Howrah"]
for s in samples:
    print(f"{s:<28} -> {read_coordinate(s)}")

22.5726, 88.3639             -> (22.5726, 88.3639)
22.5726N 88.3639E            -> (22.5726, 88.3639)
Lat: 22.5726 Lon: 88.3639    -> (22.5726, 88.3639)
22°34'21"N 88°21'50"E        -> (22.5725, 88.36389)
somewhere near Howrah        -> None


Let us understand the code above:
- `-?\d{1,3}\.\d+` matches a decimal number, with an optional minus sign.
- `[NS]?` and `[EW]?` allow an optional direction letter.
- The DMS pattern captures degrees, minutes, seconds and the direction letter as four groups.
- If no pattern matches, the function returns `None`.

## **Example from Remote Sensing: Checking a Sentinel-2 Tile Name**
A Sentinel-2 tile name has a fixed format: **2 digits** (UTM zone) + **1 letter** (latitude band) + **2 letters** (square). For example `45QXF`.

The whole of West Bengal falls in **UTM zone 45** (which covers longitudes 84°E to 90°E). We will learn about UTM on Day 90.

In [5]:
tile = re.compile(r"^T?(?P<zone>\d{2})(?P<band>[C-X])(?P<square>[A-Z]{2})$")

for t in ["45QXF", "T45QXE", "44QPM", "45qxf", "4QXF"]:
    m = tile.match(t)
    if m:
        print(f"{t:<7} valid   zone {m['zone']}, latitude band {m['band']}, square {m['square']}")
    else:
        print(f"{t:<7} invalid")

45QXF   valid   zone 45, latitude band Q, square XF
T45QXE  valid   zone 45, latitude band Q, square XE
44QPM   valid   zone 44, latitude band Q, square PM
45qxf   invalid
4QXF    invalid


## **Password Strength with Lookahead**
`(?=...)` checks that something exists **somewhere ahead**, without using up any characters. We can stack many such checks.

In [6]:
strong = re.compile(r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d)(?=.*[@#$%&!]).{8,}$")

for pw in ["kolkata123", "Kolkata@2024", "SHORT1!", "Mishti#Doi9"]:
    print(f"{pw:<14}{'strong' if strong.match(pw) else 'weak'}")

kolkata123    weak
Kolkata@2024  strong
SHORT1!       weak
Mishti#Doi9   strong


## **Cleaning Text**

In [7]:
messy = "  Kolkata   is   the    City  of   Joy!!!   "
print(re.sub(r"\s+", " ", messy).strip())

bengali = "আমার সোনার বাংলা, আমি তোমায় ভালোবাসি।"
print(re.findall(r"\w+", bengali))      # \w also matches Bengali letters

Kolkata is the City of Joy!!!
['আম', 'র', 'স', 'ন', 'র', 'ব', 'ল', 'আম', 'ত', 'ম', 'য', 'ভ', 'ল', 'ব', 'স']


## **Tip**
Test your patterns on the website **regex101.com** before using them in code. It explains every part of the pattern.

In [8]:
# Quick question
# Which of these match the pattern ^WB\d{2}$ ?
# "WB02", "WB2", "wb02", "WB021"